In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [5]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 

In [ ]:
param_grid = {
    "model__C": np.logspace(-3, 2, 6), # log scale search over 10^-3, 10^2, 6 splits: 0.001, 0.01, 0.1, 1, 10, 100. Smaller C = stronger regulisation
    "model__penalty": ["elasticnet"] # balance between L1 and L2 regulisation
}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42, solver = "saga")) # saga required for elasticnet
])

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=skf,
    scoring="roc_auc",
    refit=True,        # refit best model on full training set
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f"Best C:       {grid_search.best_params_['model__C']}")
print(f"Best AUC-ROC: {grid_search.best_score_:.4f}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results = results[[
    "param_model__C",
    "mean_test_score",
    "std_test_score",
    "mean_train_score"
]].copy()

results.columns = ["C", "mean_auc_roc", "std_auc_roc", "mean_train_auc"]
results["C"] = results["C"].astype(float)
results = results.sort_values("C").reset_index(drop=True)

print(results.to_string(index=False))